# Parte 4 — Pandas en la práctica: el DataFrame
### Recorrido guiado (con datos reales) sobre `clase_fundamentos_python_CD.md`, sección 4

Un **DataFrame** es la estructura central de pandas: se puede pensar como una hoja de Excel dentro de Python (filas = registros, columnas = variables). A continuación se recorre cada subsección de la Parte 4 usando el archivo `covid_19_data.csv` de esta misma carpeta.

## 4.1 Leer datos
El archivo real se llama `covid_19_data.csv` (en las notas de clase aparece como `covid_data.csv`, solo ajusta el nombre al del archivo que tengas).

In [ ]:
import pandas as pd

data = pd.read_csv("covid_19_data.csv")
data.head()

Otros formatos que pandas puede leer: `pd.read_excel()`, `pd.read_json()`, `pd.read_html()`, o conexión directa a bases de datos con `pd.read_sql()`.

## 4.2 Exploración inicial

In [ ]:
print(data.head())      # primeros 5 registros
print(data.tail())      # últimos 5 registros
print(data.shape)       # (número de filas, número de columnas)

In [ ]:
data.info()      # tipo de dato por columna, nulos, memoria usada

In [ ]:
data.describe()  # estadísticas básicas (media, min, max, percentiles...)

## 4.3 Tipos de dato y fechas
Por default, pandas lee las columnas de fecha como texto (`object`). Nota que arriba, en `data.info()`, `ObservationDate` y `Last Update` aparecen como `object`, no como fecha. Para poder ordenar/filtrar por fecha, hay que indicarle explícitamente a `read_csv` cuáles columnas son fechas:

In [ ]:
data = pd.read_csv("covid_19_data.csv", parse_dates=["ObservationDate", "Last Update"])
data.info()   # ahora esas columnas aparecen como datetime64

**Por qué importa:** solo se pueden hacer operaciones temporales (ordenar cronológicamente, filtrar por rango de fechas, graficar series de tiempo) si el tipo de dato es `datetime`, no texto.

## 4.4 Selección de columnas y valores únicos

In [ ]:
data["Country/Region"]              # selecciona una columna (Serie de pandas)

In [ ]:
data["Country/Region"].unique()     # lista de valores distintos en esa columna

Útil para: explorar categorías disponibles, y verificar que un filtro funcionó correctamente.

## 4.5 Filtrado de filas

In [ ]:
china = data[data["Country/Region"] == "Mainland China"]
mexico = data[data["Country/Region"] == "Mexico"]

print(china["Country/Region"].unique())
print(mexico["Country/Region"].unique())

**Buena práctica de exploración:** después de filtrar, verificar con `.unique()` que solo quedó lo esperado, y revisar con `.head()`/`.tail()` el resultado.

In [ ]:
print(china.head())
print(china.tail())

print("Provincias/estados en China:", china["Province/State"].unique())
print("Provincias/estados en México:", mexico["Province/State"].unique())

Aquí se ve la diferencia de granularidad mencionada en las notas: China reporta por provincia/estado, mientras que México solo tiene un registro por país (sin desagregar). Esto importa antes de decidir a qué nivel se puede hacer el análisis.

## 4.6 Agrupar y agregar (`groupby`)

In [ ]:
data.groupby("Country/Region")["Confirmed"].max()

Esto responde: *"para cada país, ¿cuál fue el número máximo de confirmados registrado en un solo día?"*. Otras funciones de agregación: `.sum()`, `.min()`, `.mean()`, `.count()`.

`groupby` cambia la estructura de índices del resultado (el país queda como índice, no como columna); `.reset_index()` regresa el resultado a un DataFrame "plano", fácil de seguir usando con la notación habitual (`casos_pais["Country/Region"]`, por ejemplo).

In [ ]:
casos_pais = data.groupby("Country/Region")["Confirmed"].max().reset_index()
casos_pais

## 4.7 Ordenar (`sort_values`)

In [ ]:
casos_pais.sort_values("Confirmed", ascending=False)   # de mayor a menor

In [ ]:
casos_pais.sort_values("Confirmed", ascending=True)    # de menor a mayor (default)

## 4.8 Tablas dinámicas (`pivot_table`)

**Nota sobre esta versión de pandas:** el ejemplo de las notas de clase usa `aggfunc="sum"` sin indicar `values`, lo que hace que pandas intente sumar *todas* las columnas numéricas y de fecha del DataFrame. En versiones recientes de pandas esto produce un error (`TypeError: datetime64 type does not support operation 'sum'`), porque no tiene sentido "sumar" fechas. La solución es indicar explícitamente qué columnas se quieren agregar con el parámetro `values`.

In [ ]:
# Código literal de las notas de clase -> falla en esta versión de pandas
try:
    resumen = data.pivot_table(
        index=["Country/Region", "Province/State"],
        aggfunc="sum"
    )
except TypeError as e:
    print("Error esperado:", e)

In [ ]:
# Versión corregida: se indican explícitamente las columnas numéricas a sumar
resumen = data.pivot_table(
    index=["Country/Region", "Province/State"],
    values=["Confirmed", "Deaths", "Recovered"],
    aggfunc="sum"
)
resumen

El **índice** (index) es la estructura que pandas usa para localizar información eficientemente (no hay que recorrer fila por fila). `pivot_table` permite reorganizar y agregar datos según distintos niveles (país, provincia, fecha, etc.) según lo que necesite el análisis.

Para filtrar dentro de un índice de varios niveles (MultiIndex), se usa `.loc`:

In [ ]:
resumen.loc["Mainland China"]

**Otro detalle importante encontrado con datos reales:** las notas de clase usan `resumen.loc[\"Mexico\"]` como ejemplo, pero con este dataset eso lanza `KeyError: 'Mexico'`. La razón no es un error de pandas, sino de los datos: `pivot_table` descarta las filas cuyo valor en una columna del índice es `NaN`, y México (igual que Francia) no reporta `Province/State` (todas sus filas tienen `NaN` en esa columna), así que desaparece por completo del índice de `resumen`. Es justo el punto que menciona la sección 4.5: algunos países reportan por provincia/estado y otros no, y eso cambia a qué nivel se puede analizar cada uno. Se puede comprobar así:

In [ ]:
print(data[data["Country/Region"] == "Mexico"]["Province/State"].unique())
print("Mexico" in resumen.index.get_level_values(0))

---
### Resumen de la Parte 4
- `pd.read_csv(..., parse_dates=[...])` para que las fechas queden como `datetime`, no texto.
- `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()` para explorar un DataFrame antes de transformarlo.
- `data["columna"].unique()` para ver categorías disponibles y verificar filtros.
- `data[data["columna"] == valor]` para filtrar filas.
- `groupby(...).agg().reset_index()` para agrupar y volver a un DataFrame plano.
- `sort_values(...)` para ordenar.
- `pivot_table(..., values=[...], aggfunc=...)` para tablas dinámicas — siempre indicando `values` explícitamente cuando el DataFrame tiene columnas de fecha, para evitar el error de tipo visto arriba.